In [0]:
# Databricks notebook source
"""
Buzzmetrix topic-pool pipeline for Databricks.

Pipeline
--------
1. Extract deterministic sampled reviews by
   (cate_1_depth, cate_2_depth, sc_measurement).
2. Use LLM to translate mentally into Korean and collect topic ideas only.
3. Consolidate topic ideas into a fixed topic pool per group.
4. Classify all eligible rows with fixed topic pool.
5. Collapse sparse sub topics into 기타(...).

Key rules
---------
- sentiment uses sc_measurement directly: 1=긍정, 0=중립, -1=부정
- simple praise/blame goes only to 단순 긍정 / 단순 부정
- main topic must be noun style
- sub topic must be concise Korean statement in "~~이 ~~함" style
- sparse sub topics:
  - if topic-group total <= 1000: sub topic cnt < 5 -> 기타(...)
  - if topic-group total > 1000: sub topic cnt < 10 -> 기타(...)
"""

from __future__ import annotations

import json
import re
import time
import hashlib
from typing import Any, Dict, List, Optional, Tuple

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F


# =========================
# Manual Config
# =========================
SNAPSHOT_QUARTER = "2026Q1"
PROMPT_VERSION = "v6.1.0_topic_pool"
ENDPOINT = "databricks-gpt-5-2"
RUN_SCOPE = "PROD"

TEST_MODE = False
TEST_MAX_GROUPS = 30
RUN_FULL_AFTER_TEST = False

TEST_CATE_1_DEPTH = "07. 스마트 사용성"
TEST_CATE_2_DEPTH = "07-06. 리모컨 사용성"

RATE_LIMIT_SECONDS = 0.6
MAX_TOKENS = 1600
MAX_RETRIES = 4
BACKOFF_BASE = 1.8

GROUP_DIMS_TO_RUN: List[str] = []  # [] 이면 전체
TARGET_Y_FEATURE = ""  # optional cate_2_depth filter

PVAL_MAX = 0.10
ABS_COEF_THRESHOLD = 0.10
USE_IS_DRIVER = True

MAX_DRIVERS_PER_KEY_FULL = 4
MAX_DRIVERS_PER_KEY_COMPACT = 3
MAX_DIFF_STATS_FULL = 10

SOURCE_TABLE = "kic_data_ods.buzzmetrix.buzzmetrix"
LABELED_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_labeled_topic_table_{PROMPT_VERSION.replace('.', '_')}"
TOPIC_POOL_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_topic_pool_{PROMPT_VERSION.replace('.', '_')}"
TOPIC_IDEA_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_topic_idea_{PROMPT_VERSION.replace('.', '_')}"
CANDIDATE_REVIEW_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_candidate_topic_review_{PROMPT_VERSION.replace('.', '_')}"
APPROVED_TOPIC_POOL_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_topic_pool_approved_{PROMPT_VERSION.replace('.', '_')}"
FAILED_TABLE = f"sandbox.z_jungryo_lee.buzzmetrix_topic_failed_{PROMPT_VERSION.replace('.', '_')}"

SEED = "seed_20260416"
MIN_GROUP_SIZE = 100
MAX_SAMPLE_ROWS_PER_GROUP = 1000
STAGE1_CHUNK_SIZE = 80
STAGE2_BATCH_SIZE = 1
TOPIC_POOL_TARGET_MIN = 12
TOPIC_POOL_TARGET_MAX = 15
MAX_RARE_SUBTOPIC_EXAMPLES = 3
MAX_FAILED_RETRIES = 2

print(
    f"[CONFIG] quarter={SNAPSHOT_QUARTER}, prompt_version={PROMPT_VERSION}, "
    f"endpoint={ENDPOINT}, run_scope={RUN_SCOPE}, test_mode={TEST_MODE}"
)
print(
    f"[CONFIG] test_cate_1={TEST_CATE_1_DEPTH}, test_cate_2={TEST_CATE_2_DEPTH}, "
    f"test_max_groups={TEST_MAX_GROUPS}, target_y_feature={TARGET_Y_FEATURE or 'ALL'}"
)


# =========================
# Utilities
# =========================
def clean_text(x: Any) -> str:
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def normalize_text(x: Any) -> str:
    return clean_text(x).replace("\u200b", " ")


def md5_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def sc_to_sentiment(sc_value: Any) -> str:
    try:
        value = int(sc_value)
    except Exception:
        return "기타"
    if value == 1:
        return "긍정"
    if value == 0:
        return "중립"
    if value == -1:
        return "부정"
    return "기타"


def safe_list(value: Any) -> List[Any]:
    return value if isinstance(value, list) else []


def chunk_list(items: List[Any], size: int) -> List[List[Any]]:
    return [items[i:i + size] for i in range(0, len(items), size)]


def build_group_key(cate_1_depth: str, cate_2_depth: str, sc_measurement: Any) -> str:
    return f"{clean_text(cate_1_depth)}||{clean_text(cate_2_depth)}||{clean_text(sc_measurement)}"


def table_name_for_mode(base_name: str, test_mode: bool) -> str:
    return f"{base_name}_test" if test_mode else base_name


def extract_json_object(text: str) -> Dict[str, Any]:
    raw = clean_text(text)
    try:
        return json.loads(raw)
    except Exception:
        pass

    fenced = re.search(r"```json\s*(\{.*\})\s*```", raw, flags=re.S)
    if fenced:
        return json.loads(fenced.group(1))

    match = re.search(r"\{.*\}", raw, flags=re.S)
    if match:
        candidate = match.group(0)
        candidate = candidate.replace("\u201c", '"').replace("\u201d", '"')
        candidate = candidate.replace("\u2018", "'").replace("\u2019", "'")
        candidate = re.sub(r",\s*}", "}", candidate)
        candidate = re.sub(r",\s*]", "]", candidate)
        return json.loads(candidate)

    raise ValueError(f"Invalid JSON from model: {raw[:500]}")


def call_endpoint(messages: List[Dict[str, str]]) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client

    client = get_deploy_client("databricks")
    payload = {
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": MAX_TOKENS,
    }

    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.predict(endpoint=ENDPOINT, inputs=payload)
            if isinstance(resp, dict):
                if "choices" in resp and resp["choices"]:
                    return extract_json_object(resp["choices"][0]["message"]["content"])
                if "predictions" in resp and resp["predictions"]:
                    pred0 = resp["predictions"][0]
                    if isinstance(pred0, dict) and "content" in pred0:
                        return extract_json_object(pred0["content"])
                    if isinstance(pred0, str):
                        return extract_json_object(pred0)
                if "content" in resp:
                    return extract_json_object(resp["content"])
            if isinstance(resp, str):
                return extract_json_object(resp)
            raise ValueError(f"Unexpected response schema: {type(resp)}")
        except Exception as exc:
            last_error = exc
            try:
                repair_payload = {
                    "messages": messages + [
                        {"role": "assistant", "content": "이전 응답이 JSON 형식에 맞지 않았습니다."},
                        {
                            "role": "user",
                            "content": (
                                "설명 없이 JSON 객체 하나만 다시 출력하세요. "
                                "마크다운, 코드블록, 부가 문장 없이 순수 JSON만 반환하세요."
                            ),
                        },
                    ],
                    "temperature": 0.0,
                    "max_tokens": MAX_TOKENS,
                }
                resp = client.predict(endpoint=ENDPOINT, inputs=repair_payload)
                if isinstance(resp, dict):
                    if "choices" in resp and resp["choices"]:
                        return extract_json_object(resp["choices"][0]["message"]["content"])
                    if "predictions" in resp and resp["predictions"]:
                        pred0 = resp["predictions"][0]
                        if isinstance(pred0, dict) and "content" in pred0:
                            return extract_json_object(pred0["content"])
                        if isinstance(pred0, str):
                            return extract_json_object(pred0)
                    if "content" in resp:
                        return extract_json_object(resp["content"])
                if isinstance(resp, str):
                    return extract_json_object(resp)
            except Exception as repair_exc:
                last_error = repair_exc
            print(f"[LLM ERROR] attempt={attempt + 1}/{MAX_RETRIES}, error={repr(last_error)}")
            time.sleep(BACKOFF_BASE ** attempt)

    raise RuntimeError(f"Endpoint request failed after retries: {last_error}")


def save_table(df: DataFrame, table_name: str) -> None:
    target = table_name
    if spark.catalog.tableExists(target):
        print(f"[SAVE] overwrite -> {target}")
    else:
        print(f"[SAVE] create -> {target}")
    df.write.mode("overwrite").format("delta").saveAsTable(target)


def empty_failed_df() -> DataFrame:
    schema = """
        review_id string,
        cate_1_depth string,
        cate_2_depth string,
        sc_measurement int,
        sentiment string,
        stage string,
        endpoint string,
        error_message string,
        payload_key string,
        memo string
    """
    return spark.createDataFrame([], schema)


# =========================
# Data Extraction
# =========================
def build_sample_query(test_mode: bool) -> str:
    extra_filters: List[str] = []
    if TARGET_Y_FEATURE:
        extra_filters.append(f"cate_2_depth = '{TARGET_Y_FEATURE}'")
    if test_mode:
        extra_filters.append(f"cate_1_depth = '{TEST_CATE_1_DEPTH}'")
        extra_filters.append(f"cate_2_depth = '{TEST_CATE_2_DEPTH}'")
    if GROUP_DIMS_TO_RUN:
        group_filter = ", ".join([f"'{x}'" for x in GROUP_DIMS_TO_RUN])
        extra_filters.append(f"concat(cate_1_depth, '||', cate_2_depth, '||', cast(sc_measurement as string)) in ({group_filter})")

    extra_where = ""
    if extra_filters:
        extra_where = "\n      and " + "\n      and ".join(extra_filters)

    return f"""
with base as (
    select distinct
        cate_1_depth,
        cate_2_depth,
        sc_measurement,
        memo
    from {SOURCE_TABLE}
    where 1=1
      and cate_1_depth not like '***%%'
      and memo is not null
      and length(trim(memo)) > 0
      {extra_where}
),
sampled as (
    select
        cate_1_depth,
        cate_2_depth,
        sc_measurement,
        memo,
        count(*) over (
            partition by cate_1_depth, cate_2_depth, sc_measurement
        ) as grp_cnt,
        row_number() over (
            partition by cate_1_depth, cate_2_depth, sc_measurement
            order by md5(
                concat(
                    coalesce(cate_1_depth, ''), '||',
                    coalesce(cate_2_depth, ''), '||',
                    coalesce(sc_measurement, ''), '||',
                    coalesce(memo, ''), '||',
                    '{SEED}'
                )
            )
        ) as rn
    from base
)
select
    sha2(concat_ws('||', cate_1_depth, cate_2_depth, cast(sc_measurement as string), memo), 256) as review_id,
    cate_1_depth,
    cate_2_depth,
    cast(sc_measurement as int) as sc_measurement,
    memo,
    grp_cnt
from sampled
where grp_cnt >= {MIN_GROUP_SIZE}
  and rn <= {MAX_SAMPLE_ROWS_PER_GROUP}
order by cate_1_depth, cate_2_depth, sc_measurement, rn
"""


def build_full_query(test_mode: bool) -> str:
    extra_filters: List[str] = []
    if TARGET_Y_FEATURE:
        extra_filters.append(f"cate_2_depth = '{TARGET_Y_FEATURE}'")
    if test_mode:
        extra_filters.append(f"cate_1_depth = '{TEST_CATE_1_DEPTH}'")
        extra_filters.append(f"cate_2_depth = '{TEST_CATE_2_DEPTH}'")
    if GROUP_DIMS_TO_RUN:
        group_filter = ", ".join([f"'{x}'" for x in GROUP_DIMS_TO_RUN])
        extra_filters.append(f"concat(cate_1_depth, '||', cate_2_depth, '||', cast(sc_measurement as string)) in ({group_filter})")

    extra_where = ""
    if extra_filters:
        extra_where = "\n  and " + "\n  and ".join(extra_filters)

    return f"""
with base as (
    select distinct
        cate_1_depth,
        cate_2_depth,
        cast(sc_measurement as int) as sc_measurement,
        memo
    from {SOURCE_TABLE}
    where 1=1
      and cate_1_depth not like '***%%'
      and memo is not null
      and length(trim(memo)) > 0
      {extra_where}
),
eligible as (
    select
        *,
        count(*) over (
            partition by cate_1_depth, cate_2_depth, sc_measurement
        ) as grp_cnt
    from base
)
select
    sha2(concat_ws('||', cate_1_depth, cate_2_depth, cast(sc_measurement as string), memo), 256) as review_id,
    cate_1_depth,
    cate_2_depth,
    sc_measurement,
    memo,
    grp_cnt
from eligible
where grp_cnt >= {MIN_GROUP_SIZE}
"""


def load_group_scope_df(test_mode: bool) -> DataFrame:
    sample_df = spark.sql(build_sample_query(test_mode))
    if test_mode:
        groups = (
            sample_df.select("cate_1_depth", "cate_2_depth", "sc_measurement")
            .distinct()
            .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
            .limit(TEST_MAX_GROUPS)
        )
        sample_df = sample_df.join(groups, ["cate_1_depth", "cate_2_depth", "sc_measurement"], "inner")
    return (
        sample_df.withColumn("sentiment", F.udf(sc_to_sentiment, "string")(F.col("sc_measurement")))
        .withColumn("group_key", F.concat_ws("||", "cate_1_depth", "cate_2_depth", F.col("sc_measurement").cast("string")))
    )


def load_full_df(test_mode: bool, group_scope_df: DataFrame) -> DataFrame:
    full_df = spark.sql(build_full_query(test_mode))
    groups = group_scope_df.select("cate_1_depth", "cate_2_depth", "sc_measurement").distinct()
    return (
        full_df.join(groups, ["cate_1_depth", "cate_2_depth", "sc_measurement"], "inner")
        .withColumn("sentiment", F.udf(sc_to_sentiment, "string")(F.col("sc_measurement")))
        .withColumn("group_key", F.concat_ws("||", "cate_1_depth", "cate_2_depth", F.col("sc_measurement").cast("string")))
    )


# =========================
# Prompts
# =========================
def build_stage1_chunk_messages(
    cate_1_depth: str,
    cate_2_depth: str,
    sentiment: str,
    memos: List[str],
) -> List[Dict[str, str]]:
    system = f"""
당신은 VOC 후보 주제 발굴 전문가다.
리뷰가 다국어이면 먼저 한국어로 번역해 이해한 뒤 분석하라.

현재 고정 그룹:
- cate_1_depth = {cate_1_depth}
- cate_2_depth = {cate_2_depth}
- sentiment = {sentiment}

규칙:
1. 최종 taxonomy를 확정하지 말고 후보 주제 아이디어만 수집한다.
2. 대주제는 반드시 명사형으로 작성한다.
3. 세부주제는 반드시 짧고 간단한 문장형으로 작성하고, "~~이 ~~함" 스타일을 사용한다.
4. 단순한 칭찬/불만은 '단순 긍정' 또는 '단순 부정' 후보로만 제안한다.
5. '단순 긍정', '단순 부정'은 정말 짧고 이유가 없는 단문에만 허용한다.
6. 서비스, 기능, 사용 상황, 원인, 비교가 보이면 절대 단순 평가로 두지 말고 실제 속성/기능 대주제로 제안한다.
7. 설명 필드는 만들지 않는다.
8. JSON 객체 하나만 반환한다.

출력 스키마:
{{
  "candidates": [
    {{
      "main_topic": "대주제",
      "sub_topic": "세부주제",
      "example_review_ko": "대표 예시 한국어",
      "example_review_original": "대표 예시 원문"
    }}
  ]
}}
"""
    review_lines = [{"memo": clean_text(memo)} for memo in memos]
    user = f"아래 리뷰에서 후보 주제를 수집하라.\n{json.dumps(review_lines, ensure_ascii=False)}"
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_stage1_consolidation_messages(
    cate_1_depth: str,
    cate_2_depth: str,
    sentiment: str,
    candidates: List[Dict[str, Any]],
) -> List[Dict[str, str]]:
    system = f"""
당신은 VOC taxonomy 설계 전문가다.
후보 주제를 정리해 최종 고정 topic pool 초안을 설계하라.

현재 고정 그룹:
- cate_1_depth = {cate_1_depth}
- cate_2_depth = {cate_2_depth}
- sentiment = {sentiment}

규칙:
1. 최종 대주제 개수는 {TOPIC_POOL_TARGET_MIN}~{TOPIC_POOL_TARGET_MAX}개로 정리한다.
2. 대주제는 반드시 명사형으로 작성한다.
3. 세부주제는 반드시 간단명료한 "~~이 ~~함" 스타일로 작성한다.
4. '단순 긍정' 또는 '단순 부정'은 정말 단순 문장만 받도록 좁게 설계한다.
5. 가능한 한 '기타'는 남발하지 않는다.
6. 각 대주제에 1개 이상 세부주제를 둔다.
7. 출력은 JSON 객체 하나만 반환한다.

출력 스키마:
{{
  "topic_pool": [
    {{
      "main_topic": "대주제",
      "sub_topics": ["세부주제1", "세부주제2"],
      "simple_eval_only": "Y|N"
    }}
  ]
}}
"""
    user = f"아래 후보를 통합해 고정 topic pool 초안을 만들라.\n{json.dumps(candidates, ensure_ascii=False)}"
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def build_stage2_messages(
    row: Dict[str, Any],
    topic_pool: List[Dict[str, Any]],
) -> List[Dict[str, str]]:
    system = """
당신은 VOC 주제 분류기다.
리뷰가 다국어이면 먼저 한국어로 번역해 이해한 뒤 분류하라.

규칙:
1. sentiment는 이미 확정값이므로 다시 판단하지 않는다.
2. 대주제는 반드시 주어진 topic_pool 안의 main_topic 중 하나만 사용한다.
3. 세부주제는 가능한 topic_pool 안의 sub_topics 중 하나를 사용한다.
4. 대주제는 명사형이어야 한다.
5. 세부주제는 짧고 간결한 "~~이 ~~함" 스타일이어야 한다.
6. 단순 긍정/단순 부정은 아주 짧고 구체 이유 없는 평가에만 허용한다.
7. 기능, 서비스, 원인, 상황, 비교 정보가 보이면 단순 평가로 분류하지 않는다.
8. 출력은 JSON 객체 하나만 반환한다.

출력 스키마:
{
  "memo_ko": "한국어 번역문",
  "main_topic": "대주제",
  "sub_topic": "세부주제",
  "is_simple_eval": "Y|N",
  "evidence": "짧은 근거"
}
"""
    user = (
        f"topic_pool:\n{json.dumps(topic_pool, ensure_ascii=False)}\n\n"
        f"감성값: {row['sentiment']}\n"
        f"cate_1_depth: {row['cate_1_depth']}\n"
        f"cate_2_depth: {row['cate_2_depth']}\n"
        f"원문 리뷰:\n{row['memo']}"
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


# =========================
# Stage 1
# =========================
def collect_topic_ideas(sample_df: DataFrame) -> Tuple[DataFrame, DataFrame]:
    rows = [r.asDict() for r in sample_df.orderBy("cate_1_depth", "cate_2_depth", "sc_measurement").toLocalIterator()]

    grouped: Dict[str, List[Dict[str, Any]]] = {}
    for row in rows:
        grouped.setdefault(row["group_key"], []).append(row)

    topic_idea_rows: List[Dict[str, Any]] = []
    candidate_review_rows: List[Dict[str, Any]] = []
    failed_rows: List[Dict[str, Any]] = []

    for group_key, group_rows in grouped.items():
        head = group_rows[0]
        chunks = chunk_list(group_rows, STAGE1_CHUNK_SIZE)
        print(f"[STAGE1-IDEA] group={group_key}, rows={len(group_rows)}, chunks={len(chunks)}")

        for chunk_idx, chunk in enumerate(chunks, start=1):
            memos = [clean_text(x["memo"]) for x in chunk]
            messages = build_stage1_chunk_messages(
                cate_1_depth=head["cate_1_depth"],
                cate_2_depth=head["cate_2_depth"],
                sentiment=head["sentiment"],
                memos=memos,
            )
            try:
                result = call_endpoint(messages)
                for cand in safe_list(result.get("candidates")):
                    main_topic = clean_text(cand.get("main_topic"))
                    sub_topic = clean_text(cand.get("sub_topic"))
                    if not main_topic or not sub_topic:
                        continue
                    topic_idea_rows.append(
                        {
                            "group_key": group_key,
                            "cate_1_depth": head["cate_1_depth"],
                            "cate_2_depth": head["cate_2_depth"],
                            "sc_measurement": head["sc_measurement"],
                            "sentiment": head["sentiment"],
                            "chunk_id": chunk_idx,
                            "main_topic": main_topic,
                            "sub_topic": sub_topic,
                            "example_review_ko": clean_text(cand.get("example_review_ko")),
                            "example_review_original": clean_text(cand.get("example_review_original")),
                        }
                    )
                for item in chunk:
                    candidate_review_rows.append(
                        {
                            "group_key": group_key,
                            "cate_1_depth": item["cate_1_depth"],
                            "cate_2_depth": item["cate_2_depth"],
                            "sc_measurement": item["sc_measurement"],
                            "sentiment": item["sentiment"],
                            "review_id": item["review_id"],
                            "memo": item["memo"],
                        }
                    )
            except Exception as exc:
                failed_rows.append(
                    {
                        "review_id": "",
                        "cate_1_depth": head["cate_1_depth"],
                        "cate_2_depth": head["cate_2_depth"],
                        "sc_measurement": int(head["sc_measurement"]),
                        "sentiment": head["sentiment"],
                        "stage": "stage1_idea",
                        "endpoint": ENDPOINT,
                        "error_message": clean_text(exc),
                        "payload_key": f"{group_key}::chunk{chunk_idx}",
                        "memo": "",
                    }
                )

    idea_schema = """
        group_key string,
        cate_1_depth string,
        cate_2_depth string,
        sc_measurement int,
        sentiment string,
        chunk_id int,
        main_topic string,
        sub_topic string,
        example_review_ko string,
        example_review_original string
    """
    review_schema = """
        group_key string,
        cate_1_depth string,
        cate_2_depth string,
        sc_measurement int,
        sentiment string,
        review_id string,
        memo string
    """

    topic_idea_df = spark.createDataFrame(topic_idea_rows, idea_schema) if topic_idea_rows else spark.createDataFrame([], idea_schema)
    candidate_review_df = spark.createDataFrame(candidate_review_rows, review_schema) if candidate_review_rows else spark.createDataFrame([], review_schema)
    failed_df = spark.createDataFrame(failed_rows, empty_failed_df().schema) if failed_rows else empty_failed_df()
    return topic_idea_df, candidate_review_df, failed_df


def consolidate_topic_pool(topic_idea_df: DataFrame) -> Tuple[DataFrame, DataFrame]:
    idea_rows = [r.asDict() for r in topic_idea_df.orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", "main_topic", "sub_topic").toLocalIterator()]
    grouped: Dict[str, List[Dict[str, Any]]] = {}
    for row in idea_rows:
        grouped.setdefault(row["group_key"], []).append(row)

    pool_rows: List[Dict[str, Any]] = []
    failed_rows: List[Dict[str, Any]] = []

    for group_key, rows in grouped.items():
        head = rows[0]
        candidates = [
            {
                "main_topic": x["main_topic"],
                "sub_topic": x["sub_topic"],
                "example_review_ko": x["example_review_ko"],
                "example_review_original": x["example_review_original"],
            }
            for x in rows
        ]
        try:
            result = call_endpoint(
                build_stage1_consolidation_messages(
                    cate_1_depth=head["cate_1_depth"],
                    cate_2_depth=head["cate_2_depth"],
                    sentiment=head["sentiment"],
                    candidates=candidates,
                )
            )
            topic_pool = safe_list(result.get("topic_pool"))
            for major_order, topic in enumerate(topic_pool, start=1):
                main_topic = clean_text(topic.get("main_topic"))
                sub_topics = [clean_text(x) for x in safe_list(topic.get("sub_topics")) if clean_text(x)]
                if not main_topic or not sub_topics:
                    continue
                for sub_order, sub_topic in enumerate(sub_topics, start=1):
                    pool_rows.append(
                        {
                            "group_key": group_key,
                            "cate_1_depth": head["cate_1_depth"],
                            "cate_2_depth": head["cate_2_depth"],
                            "sc_measurement": head["sc_measurement"],
                            "sentiment": head["sentiment"],
                            "major_order": major_order,
                            "main_topic": main_topic,
                            "sub_order": sub_order,
                            "sub_topic": sub_topic,
                            "simple_eval_only": clean_text(topic.get("simple_eval_only")) or "N",
                        }
                    )
        except Exception as exc:
            failed_rows.append(
                {
                    "review_id": "",
                    "cate_1_depth": head["cate_1_depth"],
                    "cate_2_depth": head["cate_2_depth"],
                    "sc_measurement": int(head["sc_measurement"]),
                    "sentiment": head["sentiment"],
                    "stage": "stage1_consolidation",
                    "endpoint": ENDPOINT,
                    "error_message": clean_text(exc),
                    "payload_key": group_key,
                    "memo": "",
                }
            )

    pool_schema = """
        group_key string,
        cate_1_depth string,
        cate_2_depth string,
        sc_measurement int,
        sentiment string,
        major_order int,
        main_topic string,
        sub_order int,
        sub_topic string,
        simple_eval_only string
    """
    pool_df = spark.createDataFrame(pool_rows, pool_schema) if pool_rows else spark.createDataFrame([], pool_schema)
    failed_df = spark.createDataFrame(failed_rows, empty_failed_df().schema) if failed_rows else empty_failed_df()
    return pool_df, failed_df


def build_topic_pool_lookup(pool_df: DataFrame) -> Dict[str, List[Dict[str, Any]]]:
    rows = [r.asDict() for r in pool_df.orderBy("group_key", "major_order", "sub_order").toLocalIterator()]
    grouped: Dict[str, Dict[str, Dict[str, Any]]] = {}

    for row in rows:
        group_key = row["group_key"]
        main_topic = row["main_topic"]
        grouped.setdefault(group_key, {})
        grouped[group_key].setdefault(
            main_topic,
            {
                "main_topic": main_topic,
                "sub_topics": [],
                "simple_eval_only": row["simple_eval_only"],
            },
        )
        grouped[group_key][main_topic]["sub_topics"].append(row["sub_topic"])

    final_lookup: Dict[str, List[Dict[str, Any]]] = {}
    for group_key, topic_map in grouped.items():
        final_lookup[group_key] = list(topic_map.values())
    return final_lookup


# =========================
# Stage 2
# =========================
def classify_reviews(full_df: DataFrame, topic_pool_lookup: Dict[str, List[Dict[str, Any]]]) -> Tuple[DataFrame, DataFrame]:
    rows = [r.asDict() for r in full_df.orderBy("cate_1_depth", "cate_2_depth", "sc_measurement").toLocalIterator()]

    detail_rows: List[Dict[str, Any]] = []
    failed_rows: List[Dict[str, Any]] = []

    for idx, row in enumerate(rows, start=1):
        group_key = row["group_key"]
        topic_pool = topic_pool_lookup.get(group_key, [])
        if not topic_pool:
            failed_rows.append(
                {
                    "review_id": row["review_id"],
                    "cate_1_depth": row["cate_1_depth"],
                    "cate_2_depth": row["cate_2_depth"],
                    "sc_measurement": int(row["sc_measurement"]),
                    "sentiment": row["sentiment"],
                    "stage": "stage2_missing_pool",
                    "endpoint": ENDPOINT,
                    "error_message": "topic_pool_not_found",
                    "payload_key": group_key,
                    "memo": row["memo"],
                }
            )
            continue

        success = False
        last_error = ""
        for _ in range(MAX_FAILED_RETRIES + 1):
            try:
                result = call_endpoint(build_stage2_messages(row, topic_pool))
                detail_rows.append(
                    {
                        "review_id": row["review_id"],
                        "group_key": group_key,
                        "cate_1_depth": row["cate_1_depth"],
                        "cate_2_depth": row["cate_2_depth"],
                        "sc_measurement": int(row["sc_measurement"]),
                        "sentiment": row["sentiment"],
                        "memo": row["memo"],
                        "memo_ko": clean_text(result.get("memo_ko")),
                        "main_topic": clean_text(result.get("main_topic")) or "기타",
                        "sub_topic": clean_text(result.get("sub_topic")) or "기타 이슈가 존재함",
                        "is_simple_eval": clean_text(result.get("is_simple_eval")) or "N",
                        "evidence": clean_text(result.get("evidence")),
                    }
                )
                success = True
                break
            except Exception as exc:
                last_error = clean_text(exc)
        if not success:
            failed_rows.append(
                {
                    "review_id": row["review_id"],
                    "cate_1_depth": row["cate_1_depth"],
                    "cate_2_depth": row["cate_2_depth"],
                    "sc_measurement": int(row["sc_measurement"]),
                    "sentiment": row["sentiment"],
                    "stage": "stage2_classification",
                    "endpoint": ENDPOINT,
                    "error_message": last_error,
                    "payload_key": group_key,
                    "memo": row["memo"],
                }
            )
        if idx % 100 == 0:
            print(f"[STAGE2] classified rows={idx}")
            time.sleep(RATE_LIMIT_SECONDS)

    detail_schema = """
        review_id string,
        group_key string,
        cate_1_depth string,
        cate_2_depth string,
        sc_measurement int,
        sentiment string,
        memo string,
        memo_ko string,
        main_topic string,
        sub_topic string,
        is_simple_eval string,
        evidence string
    """
    detail_df = spark.createDataFrame(detail_rows, detail_schema) if detail_rows else spark.createDataFrame([], detail_schema)
    failed_df = spark.createDataFrame(failed_rows, empty_failed_df().schema) if failed_rows else empty_failed_df()
    return detail_df, failed_df


# =========================
# Post Process
# =========================
def collapse_sparse_subtopics(detail_df: DataFrame) -> DataFrame:
    detail_df.createOrReplaceTempView("tmp_detail_before_sparse_merge")
    sql = f"""
    with base as (
        select
            *,
            count(*) over (
                partition by cate_1_depth, cate_2_depth, sentiment, main_topic
            ) as topic_group_total_cnt,
            count(*) over (
                partition by cate_1_depth, cate_2_depth, sentiment, main_topic, sub_topic
            ) as sub_topic_cnt
        from tmp_detail_before_sparse_merge
    ),
    flagged as (
        select
            *,
            case
                when main_topic in ('단순 긍정', '단순 부정') then 0
                when topic_group_total_cnt <= 1000 and sub_topic_cnt < 5 then 1
                when topic_group_total_cnt > 1000 and sub_topic_cnt < 10 then 1
                else 0
            end as rare_flag
        from base
    ),
    rare_names as (
        select
            cate_1_depth,
            cate_2_depth,
            sentiment,
            main_topic,
            concat_ws(', ', slice(sort_array(collect_set(sub_topic)), 1, {MAX_RARE_SUBTOPIC_EXAMPLES})) as rare_examples
        from flagged
        where rare_flag = 1
        group by cate_1_depth, cate_2_depth, sentiment, main_topic
    )
    select
        f.review_id,
        f.group_key,
        f.cate_1_depth,
        f.cate_2_depth,
        f.sc_measurement,
        f.sentiment,
        f.memo,
        f.memo_ko,
        f.main_topic,
        case
            when f.rare_flag = 1 and r.rare_examples is not null
                then concat('기타(', r.rare_examples, ')')
            else f.sub_topic
        end as sub_topic,
        f.is_simple_eval,
        f.evidence,
        f.topic_group_total_cnt,
        f.sub_topic_cnt
    from flagged f
    left join rare_names r
      on f.cate_1_depth = r.cate_1_depth
     and f.cate_2_depth = r.cate_2_depth
     and f.sentiment = r.sentiment
     and f.main_topic = r.main_topic
    """
    return spark.sql(sql)


def build_summary_df(detail_df: DataFrame) -> DataFrame:
    return (
        detail_df.groupBy("cate_1_depth", "cate_2_depth", "sentiment", "main_topic", "sub_topic")
        .agg(
            F.count("*").alias("review_cnt"),
            F.countDistinct("review_id").alias("review_id_cnt"),
        )
        .orderBy("cate_1_depth", "cate_2_depth", "sentiment", F.desc("review_cnt"), "main_topic", "sub_topic")
    )


# =========================
# Run
# =========================
def run_pipeline(test_mode: bool) -> Dict[str, DataFrame]:
    sample_df = load_group_scope_df(test_mode=test_mode).cache()
    print(f"[RUN] sampled rows = {sample_df.count()}")

    full_df = load_full_df(test_mode=test_mode, group_scope_df=sample_df).cache()
    print(f"[RUN] full eligible rows = {full_df.count()}")

    topic_idea_df, candidate_review_df, failed_stage1_idea_df = collect_topic_ideas(sample_df)
    print(f"[RUN] topic idea rows = {topic_idea_df.count()}")

    topic_pool_df, failed_stage1_pool_df = consolidate_topic_pool(topic_idea_df)
    print(f"[RUN] topic pool rows = {topic_pool_df.count()}")

    topic_pool_lookup = build_topic_pool_lookup(topic_pool_df)

    detail_raw_df, failed_stage2_df = classify_reviews(full_df, topic_pool_lookup)
    print(f"[RUN] raw detail rows = {detail_raw_df.count()}")

    detail_df = collapse_sparse_subtopics(detail_raw_df)
    summary_df = build_summary_df(detail_df)

    failed_df = failed_stage1_idea_df.unionByName(failed_stage1_pool_df, allowMissingColumns=True).unionByName(
        failed_stage2_df, allowMissingColumns=True
    )

    labeled_table = table_name_for_mode(LABELED_TABLE, test_mode)
    topic_pool_table = table_name_for_mode(TOPIC_POOL_TABLE, test_mode)
    topic_idea_table = table_name_for_mode(TOPIC_IDEA_TABLE, test_mode)
    candidate_review_table = table_name_for_mode(CANDIDATE_REVIEW_TABLE, test_mode)
    approved_topic_pool_table = table_name_for_mode(APPROVED_TOPIC_POOL_TABLE, test_mode)
    failed_table = table_name_for_mode(FAILED_TABLE, test_mode)

    save_table(detail_df, labeled_table)
    save_table(summary_df, topic_pool_table)
    save_table(topic_idea_df, topic_idea_table)
    save_table(candidate_review_df, candidate_review_table)
    save_table(topic_pool_df, approved_topic_pool_table)
    save_table(failed_df, failed_table)

    print("[DISPLAY] summary preview")
    display(summary_df.limit(100))
    print("[DISPLAY] labeled preview")
    display(detail_df.limit(50))

    sample_df.unpersist()
    full_df.unpersist()

    return {
        "sample_df": sample_df,
        "full_df": full_df,
        "topic_idea_df": topic_idea_df,
        "candidate_review_df": candidate_review_df,
        "topic_pool_df": topic_pool_df,
        "detail_df": detail_df,
        "summary_df": summary_df,
        "failed_df": failed_df,
    }


print("[RUN] test pass first")
test_result = run_pipeline(test_mode=True)

if RUN_FULL_AFTER_TEST:
    print("[RUN] full pass")
    full_result = run_pipeline(test_mode=False)
